In [1]:
from bs4 import BeautifulSoup

# BeautifulSoup는 언제 사용?
# requests로 요청한 결과가 html 코드일 때 BS를 이용해서 탐색 후 원하는 데이터를 추출

html = """
<h1 id="title">오늘의 명언</h1>
<div class="quote">
  <span class="text">시작이 반이다.</span>
  <small class="author">한국 속담</small>
  <a class="tag" href="/tag/start/">start</a>
  <a class="tag" href="/tag/life/">life</a>
</div>
<div class="quote">
  <span class="text">아는 것이 힘이다.</span>
  <small class="author">프랜시스 베이컨</small>
  <a class="tag" href="/tag/knowledge/">knowledge</a>
</div>
"""

soup = BeautifulSoup(html, "html.parser")

print(soup.select_one("#title").get_text()) # id가 title인 요소의 텍스트를 가져옴
quotes = soup.select("div.quote") # div태그 중 클래스가 quote은 요소를 선택
print(len(quotes))                              # 2

for q in quotes:
    text = q.select_one("span.text").get_text(strip=True)
    author = q.select_one("small.author").get_text(strip=True)
    tags = [a.get_text(strip=True) for a in q.select("a.tag")]
    links = [a["href"] for a in q.select("a.tag")]
    print(author, "|", text, "|", tags, links)

오늘의 명언
2
한국 속담 | 시작이 반이다. | ['start', 'life'] ['/tag/start/', '/tag/life/']
프랜시스 베이컨 | 아는 것이 힘이다. | ['knowledge'] ['/tag/knowledge/']


In [3]:
import requests
res = requests.get("https://kin.naver.com/search/list.naver?query=%EC%8B%AC%EB%A6%AC+%EC%83%81%EB%8B%B4", timeout=5)
res.raise_for_status()

In [7]:
html = res.text #네이버 심리 상담 지식인 검색 결과 1페이지 html 코드를 가져옴


In [11]:
soup = BeautifulSoup(html, "html.parser")
titileEls = soup.select("._searchListTitleAnchor")
titles = [el.get_text(strip=True) for el in titileEls]
print(titles)

['심리상담사 되는 법', '천안 청소년심리상담', '교육대학원상담심리진학 질문', '상담심리교육대학원 입학 문의', '기도했지만 결국 제미나이로심리상담', '심리상담사의 꿈', '심리상담.', '창원심리상담센터 추천해주세요!', '아동심리상담사', '심리상담사자격증 효율적으로']


In [12]:
links = [el["href"] for el in titileEls]
print(links)

['https://kin.naver.com/qna/detail.naver?dirId=1130&docId=495332594&answerNo=1&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=70109&docId=495315656&answerNo=2&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=110703&docId=495424520&answerNo=2&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=1130&docId=495488831&answerNo=1&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=60901&docId=495571022&answerNo=2&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=1130&docId=495481689&answerNo=3&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=1130&docId=495553353&answerNo=1&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=1111&docId=495533744&answerNo=1&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=110103&docId=495574615&answerNo=1&qb=7Ius66asIOyDgeuLtA==', 'https://kin.naver.com/qna/detail.naver?dirId=406&docId=3

In [14]:
import pandas as pd
df = pd.DataFrame({"title" : titles, "links" : links})

In [15]:
print(df)

                title                                              links
0          심리상담사 되는 법  https://kin.naver.com/qna/detail.naver?dirId=1...
1          천안 청소년심리상담  https://kin.naver.com/qna/detail.naver?dirId=7...
2      교육대학원상담심리진학 질문  https://kin.naver.com/qna/detail.naver?dirId=1...
3     상담심리교육대학원 입학 문의  https://kin.naver.com/qna/detail.naver?dirId=1...
4  기도했지만 결국 제미나이로심리상담  https://kin.naver.com/qna/detail.naver?dirId=6...
5            심리상담사의 꿈  https://kin.naver.com/qna/detail.naver?dirId=1...
6               심리상담.  https://kin.naver.com/qna/detail.naver?dirId=1...
7    창원심리상담센터 추천해주세요!  https://kin.naver.com/qna/detail.naver?dirId=1...
8             아동심리상담사  https://kin.naver.com/qna/detail.naver?dirId=1...
9      심리상담사자격증 효율적으로  https://kin.naver.com/qna/detail.naver?dirId=4...


In [17]:
def get_result_kin(url:str)->pd.DataFrame:
    res = requests.get(url, timeout=5)
    res.raise_for_status()
    
    html = res.text

    soup = BeautifulSoup(html, "html.parser")
    # url에 따라 검색하려는 요소는 달라질 수 있음
    # 아래는 지식인 검색 결과 기준
    titileEls = soup.select("._searchListTitleAnchor") 
    titles = [el.get_text(strip=True) for el in titileEls]
    links = [el["href"] for el in titileEls]
    # 두 리스트를 합쳐서 데이터 프레임으로 변환
    return pd.DataFrame({"title" : titles, "links" : links})

In [21]:
query = "공부+방법"
page = 2
print(get_result_kin(f"https://kin.naver.com/search/list.naver?query={query}&page={page}"))

                 title                                              links
0         고2 화법과 언어공부법  https://kin.naver.com/qna/detail.naver?dirId=1...
1                영어공부법  https://kin.naver.com/qna/detail.naver?dirId=1...
2                수학공부법  https://kin.naver.com/qna/detail.naver?dirId=1...
3            고등학교공부 방법  https://kin.naver.com/qna/detail.naver?dirId=1...
4          기가공부법알려주세여ㅜ  https://kin.naver.com/qna/detail.naver?dirId=1...
5               수능공부방법  https://kin.naver.com/qna/detail.naver?dirId=1...
6                고1공부법  https://kin.naver.com/qna/detail.naver?dirId=8...
7  현재 고1 학생입니다공부법알려주세요  https://kin.naver.com/qna/detail.naver?dirId=1...
8           수능국어 실모공부법  https://kin.naver.com/qna/detail.naver?dirId=1...
9           수능 국어공부 방법  https://kin.naver.com/qna/detail.naver?dirId=1...


In [27]:
def get_detail_kin(url):
    res = requests.get(url, timeout=5)
    res.raise_for_status()
    
    html = res.text

    soup = BeautifulSoup(html, "html.parser")
    # url에 따라 검색하려는 요소는 달라질 수 있음
    # 아래는 지식인 검색 결과 기준
    questionDetail = soup.select_one(".questionDetail").get_text(strip=True)
    answerDetail = soup.select_one(".answerDetail").get_text(strip=True)
    
    return questionDetail, answerDetail

import time

details = []

for idx, row in df.iterrows():
    questionDetail, answerDetail = get_detail_kin(row["links"])
    details.append({"questionDetail" : questionDetail, "answerDetail" : answerDetail})
    time.sleep(1)

df_detail = pd.DataFrame(details, index=df.index)
df = df.join(df_detail)

In [28]:
print(df)

                title                                              links  \
0          심리상담사 되는 법  https://kin.naver.com/qna/detail.naver?dirId=1...   
1          천안 청소년심리상담  https://kin.naver.com/qna/detail.naver?dirId=7...   
2      교육대학원상담심리진학 질문  https://kin.naver.com/qna/detail.naver?dirId=1...   
3     상담심리교육대학원 입학 문의  https://kin.naver.com/qna/detail.naver?dirId=1...   
4  기도했지만 결국 제미나이로심리상담  https://kin.naver.com/qna/detail.naver?dirId=6...   
5            심리상담사의 꿈  https://kin.naver.com/qna/detail.naver?dirId=1...   
6               심리상담.  https://kin.naver.com/qna/detail.naver?dirId=1...   
7    창원심리상담센터 추천해주세요!  https://kin.naver.com/qna/detail.naver?dirId=1...   
8             아동심리상담사  https://kin.naver.com/qna/detail.naver?dirId=1...   
9      심리상담사자격증 효율적으로  https://kin.naver.com/qna/detail.naver?dirId=4...   

                                      questionDetail  \
0  지금 20살이고 전문대 다니고 있어요.예체능 하다가 그냥 취업해서 안정적인 월급 받...   
1  안녕하세요 고2 학생이고국가에서 해주는 oo시상담센터 이런 곳 보다 개인으로 운영하..